# Forecasting with time series foundation models

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tomMoral/2026-neurips-tutorial-tsfm/blob/main/01-forecasting.ipynb)

```
Authors: Thomas Moreau
```

A time series foundation model (TSFM) is pretrained on a large corpus of series.
It then forecasts a new series from its past values alone, with no training on it.
This is called *zero-shot* forecasting.
We use TSFMs on the hourly traffic of the Paris bike counters, from one series to every counter of the city.

1. The data: bike counters in Paris
2. A first forecast with t0
3. A zoo of architectures behind one interface
4. Evaluating on many forecast origins
5. Producing the horizon: autoregressive or direct
6. Patch size: an open design question
7. Multivariate: both directions of a street
8. Covariates known in advance: the calendar
9. Covariates known only in the past: the weather
10. Many sites: every counter of Paris
11. A year and more of history

This notebook is meant to run on a GPU. On Colab, choose a GPU runtime (*Runtime > Change runtime type*).
On a CPU, it runs a reduced version, with fewer forecast origins: the numbers are then noisier.

Set `CACHE = True` below to store every forecast in `.cache/`: the next run reuses them and loads no model,
which is handy to replay the notebook live. Delete the folder to recompute, and ignore the timings of a cached run.

In [ ]:
# Colab: clone the tutorial repository and run from its root
import os, subprocess, sys

if "google.colab" in sys.modules:
    repo = "/content/2026-neurips-tutorial-tsfm"
    if not os.path.exists(repo):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/tomMoral/2026-neurips-tutorial-tsfm", repo], check=True)
    os.chdir(repo)

In [ ]:
# !pip install -U chronos-forecasting "timesfm[torch]" tirex-2 git+https://github.com/mistralai/timestral.git "tfc-t0[torch]>=0.6.0" tabicl

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tsfm_utils as tu

warnings.filterwarnings("ignore")
CACHE = False
if CACHE:
    tu.use_cache(".cache")
GPU = tu.DEVICE == "cuda"
# On a CPU, keep one forecast origin in 4
every = 1 if GPU else 4
print("Running on", tu.DEVICE)

## 1. The data: bike counters in Paris

The city of Paris counts the bikes passing under about a hundred counters, every hour.
We start with the counter of boulevard de Sébastopol, one of the busiest cycle lanes of Paris,
from February 2020 to December 2025. The counts are in UTC.

`tu.load_bike_counts` returns one column per counter, with a regular hourly index.

In [ ]:
counts = tu.load_bike_counts()
counts.shape

In [ ]:
counter = "Totem 73 boulevard de Sébastopol S-N"
series = counts[counter]

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
series.resample("D").sum().plot(ax=axes[0], title=f"{counter}: daily total")
series["2025-06-02":"2025-06-15"].plot(ax=axes[1], title="Two weeks, hourly")
plt.tight_layout()

The series has a daily cycle with two commute peaks, a weekly cycle with quieter weekends,
and slower changes: the COVID lockdowns of 2020 and 2021, the summer and Christmas holidays,
and the growth of cycling in Paris. We forecast the year 2025.

## 2. A first forecast with t0

We forecast the next week (`horizon = 168` hours) from the last `context_length` hours.
The forecast starts at an *origin*, here a Monday at midnight.

`tu.make_windows` cuts the context before the origin and the true future after it.
It returns arrays shaped `(n_origins, n_columns, n_timesteps)`.

In [ ]:
context_length, horizon = 2048, tu.WEEK
origin = pd.Timestamp("2025-06-02")

context, future = tu.make_windows(counts[[counter]], [origin], context_length, horizon)
context, future = context[:, 0], future[:, 0]  # a single column: drop its axis
context.shape, future.shape

Our first model is **t0**, released in 2026 by The Forecasting Company, a start-up based in Paris.
Its weights are open on the Hugging Face Hub.
Like most TSFMs, it is *probabilistic*: it returns quantiles of the future values,
not a single value. Between the 10% and 90% quantiles lies an 80% prediction interval.

In [ ]:
from t0 import T0Forecaster

t0_model = T0Forecaster.from_pretrained("theforecastingcompany/t0-gamma").to(tu.DEVICE).eval()
out = t0_model.predict(context, horizon=horizon, quantile_levels=tu.QUANTILES)
out.quantiles.shape  # (n_series, horizon, n_quantiles)

In [ ]:
index = pd.date_range(end=origin + pd.Timedelta(hours=horizon - 1), periods=context_length + horizon, freq="h")
quantiles = out.quantiles.float().cpu().numpy()
tu.plot_forecast(index, context[0], future[0], quantiles[0], label="t0");

## 3. A zoo of architectures behind one interface

"Foundation model" covers very different designs. They differ on how the series is cut into
tokens, on the network, and on how the future is produced:

| Model | Backbone | Token | Output |
|---|---|---|---|
| Chronos (Amazon, 2024) | T5 encoder-decoder | one quantized value | samples, one step at a time |
| Chronos-Bolt (Amazon, 2024) | T5 encoder-decoder | patch of 16 steps | quantiles of 64 steps at once |
| Chronos-2 (Amazon, 2025) | encoder-only, attention across series | patch of 16 steps | quantiles of up to 1024 steps at once |
| TimesFM 3 (Google, 2026) | decoder-only, attention across variates | patch of 32 steps | quantiles of the whole horizon at once |
| TiRex-2 (NX-AI, 2026) | xLSTM (recurrent) | patch of 32 steps | quantiles of up to 320 steps, through the recurrent state |
| Timestral (Mistral AI, 2026) | causal transformer with forecast tokens | patch of 32 steps | quantiles of up to 1024 steps at once |
| t0 (The Forecasting Company, 2026) | decoder, time and variate attention | patch of 32 steps | quantiles of up to 1024 steps at once |
| TabICL (Inria, 2025) | tabular in-context learner | one timestep as a row of features | quantiles of every future row |

Two of them come from Paris: t0 and Timestral, the first time series models of Mistral AI.
TabICL is not a time series model: it is a foundation model for tables, developed at Inria Saclay.
It turns forecasting into a regression of the value on features of the time (calendar, detected seasons),
and learns this regression in context from the past rows.

The libraries all have different APIs. `tsfm_utils.py` wraps each of them in a class
with the same method, `predict(context, horizon)`, that returns quantiles shaped
`(n_series, horizon, n_quantiles)`. Open the file to see the native call of each library, in `forecast`.
A model loads its weights at its first forecast.
TabICL also needs the dates of the forecast origins, to build its calendar features.
We add the *seasonal naive* baseline, which repeats the last week: a TSFM is only useful if it beats it.

In [ ]:
models = {}
for model_class in [tu.SeasonalNaive, tu.Chronos, tu.ChronosBolt, tu.Chronos2,
                    tu.TimesFM3, tu.TiRex2, tu.Timestral, tu.T0, tu.TabICL]:
    model = model_class()
    models[model.name] = model
t0, chronos2, timestral, tabicl = models["t0"], models["Chronos-2"], models["Timestral"], models["TabICL"]

In [ ]:
fig, axes = plt.subplots(len(models), 1, figsize=(12, 2.2 * len(models)), sharex=True, sharey=True)
for ax, (name, model) in zip(axes, models.items()):
    kwargs = {"origins": [origin]} if model is tabicl else {}
    tu.plot_forecast(index, context[0], future[0], model.predict(context, horizon, **kwargs)[0], ax=ax, label=name)
plt.tight_layout()

<div class="alert alert-success">
    <b>QUESTION</b>:
    <ul>
    <li>Which models capture the two daily peaks? The quieter weekend?</li>
    <li>Compare the width of the prediction intervals. Which models look over- or under-confident?</li>
    </ul>
</div>

## 4. Evaluating on many forecast origins

One week is an anecdote. We repeat the forecast at many origins and average the errors:
one origin every two weeks of 2025, each with a horizon of one week.
Each origin only uses the past, so there is no leakage.

We use three metrics, standard in the TSFM benchmarks:

- **MASE** (mean absolute scaled error): the error of the median forecast, divided by the error of
  the seasonal naive forecast on the context. Below 1, the model beats "same as last week".
- **WQL** (weighted quantile loss): the pinball loss of the quantiles, divided by the total
  of the series. It scores the whole predicted distribution, not only the median.
- **coverage**: the fraction of true values inside the 80% interval. A calibrated model covers 80%.

In [ ]:
origins = pd.date_range("2025-01-06", "2025-12-15", freq="14D")[::every]
contexts, futures = tu.make_windows(counts[[counter]], origins, context_length, horizon)
contexts, futures = contexts[:, 0], futures[:, 0]

quantiles, results = {}, {}
for name, model in models.items():
    kwargs = {"origins": origins} if model is tabicl else {}
    start = time.perf_counter()
    quantiles[name] = model.predict(contexts, horizon, **kwargs)
    results[name] = {**tu.score(futures, quantiles[name], contexts), "time (s)": time.perf_counter() - start}

results = pd.DataFrame(results).T.sort_values("WQL")
results.style.format("{:.3f}")

A week is longer than the block that some models output in one call.
We look at the error and the coverage at each lead time, averaged over the origins.
The seasonal naive forecast has no interval, so it has no coverage.

In [ ]:
scale = np.mean(np.abs(contexts[:, tu.WEEK:] - contexts[:, :-tu.WEEK]), axis=1, keepdims=True)
lead_days = np.arange(horizon) // 24 + 1

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for name, q in quantiles.items():
    error = np.abs(futures - q[..., tu.MEDIAN]) / scale
    covered = (futures >= q[..., 0]) & (futures <= q[..., -1])
    axes[0].plot(pd.Series(error.mean(axis=0)).groupby(lead_days).mean(), marker="o", label=name)
    if name != "Seasonal naive":
        axes[1].plot(pd.Series(covered.mean(axis=0)).groupby(lead_days).mean(), marker="o", label=name)
axes[0].set(xlabel="lead time (days)", ylabel="scaled absolute error")
axes[1].axhline(0.8, color="k", ls=":")
axes[1].set(xlabel="lead time (days)", ylabel="coverage of the 80% interval")
axes[0].legend()
results.plot.scatter(x="time (s)", y="WQL", logx=True, ax=axes[2])
for name, row in results.iterrows():
    axes[2].annotate(name, (row["time (s)"], row["WQL"]))
plt.tight_layout()

<div class="alert alert-success">
    <b>QUESTION</b>:
    <ul>
    <li>Do the models rank in the same order with MASE and WQL? Why can they differ?</li>
    <li>Does the error grow with the lead time? What in this series makes the last days as easy as the first?</li>
    <li>Which models are over-confident? Why is Chronos so much slower than the others?</li>
    </ul>
</div>

The first call of a model includes one-off costs, such as compiling kernels: run the cell twice for a fair timing.
Each model also has its own maximum context: Chronos keeps the last 512 values of the 2048 we give.
The leaderboards of the benchmarks (GIFT-Eval, fev-bench) do not always agree with one series:
a ranking averaged over hundreds of datasets says little about yours.

## 5. Producing the horizon: autoregressive or direct

A forecaster must output a whole horizon, but a model predicts a fixed number of steps per call.
The **autoregressive** (or recursive) strategy predicts the next steps, appends them to the context
and starts again, as an AR(p) model does.
The **direct** strategy predicts every step of the horizon at once.

| | Autoregressive | Direct |
|---|---|---|
| Horizon | any | fixed when training |
| Errors | compound: each step builds on predicted values | independent across steps |
| Uncertainty | needs sampled paths to grow with the horizon | each step has its own distribution |
| Cost | one call per step | one call |

TSFMs sit between the two. Most predict a block of steps per call and roll out beyond it:

| Model | Steps per call | Beyond |
|---|---|---|
| Chronos | 1 value, sampled | 20 sampled paths, one step at a time |
| Chronos-Bolt | 64 | quantiles fed back |
| Chronos-2 | 1024 | quantiles fed back |
| TimesFM 3 | the whole horizon | none: the future patches are masked inputs |
| TiRex-2 | 320 | none: longer horizons are cut |
| Timestral | 1024 | median fed back |
| t0 | 1024 | quantiles fed back |
| TabICL | every row at once | none: each future step is a row |

To isolate the effect of the strategy, we keep one model, t0, and change only how it produces one week.
t0 can output 168 steps in one call. We can also force it to roll out:
predict a few steps, append their median to the context, and predict again.

<div class="alert alert-success">
    <b>EXERCISE</b>: autoregressive rollout
    <ul>
    <li>Write <code>rollout(model, context, horizon, step)</code>. It calls <code>model.predict</code>
    for <code>step</code> steps, appends the median forecast to the context, and repeats until the
    horizon is covered.</li>
    <li>It returns the quantiles of every step, shaped <code>(n_series, horizon, n_quantiles)</code>.</li>
    </ul>
</div>

Solution: `solutions/forecasting-1_rollout.py`

In [ ]:
def rollout(model, context, horizon, step):
    """Forecast `horizon` steps, `step` at a time, feeding the median back into the context."""
    ...

In [ ]:
forecasts = {"direct": t0.predict(contexts, horizon)}
for step in [64, 32, 8]:
    forecasts[f"rollout by {step}"] = rollout(t0, contexts, horizon, step)

pd.DataFrame({name: tu.score(futures, q, contexts) for name, q in forecasts.items()}).T

In [ ]:
index = pd.date_range(end=origins[0] + pd.Timedelta(hours=horizon - 1), periods=context_length + horizon, freq="h")
fig, axes = plt.subplots(len(forecasts), 1, figsize=(12, 9), sharex=True, sharey=True)
for ax, (name, q) in zip(axes, forecasts.items()):
    tu.plot_forecast(index, contexts[0], futures[0], q[0], ax=ax, label=f"t0, {name}")
plt.tight_layout()

<div class="alert alert-success">
    <b>QUESTION</b>:
    <ul>
    <li>How do the error and the width of the prediction interval change with the step of the rollout? Why?</li>
    <li>Feeding back the median makes the model believe its own forecast. What would sampled paths change, and at what cost?</li>
    </ul>
</div>

## 6. Patch size: an open design question

Most TSFMs cut the series into patches: a patch is one token for the transformer.
Its size sets the temporal resolution of the model, the number of tokens for a given context,
and so the cost of attention. It is fixed at pretraining, and the published models made different choices:
16 steps for Chronos-Bolt and Chronos-2, 32 for TimesFM, TiRex, Timestral and t0.

[Yehya et al. (2026)](https://arxiv.org/abs/2611.02750) trained encoder-only and decoder-only models
with several patch sizes, on the same data and with the same budget, and evaluated them on the 97 tasks of GIFT-Eval:

- encoder-only models are better with small patches,
- decoder-only models are best with an intermediate patch, and the best patch grows with the horizon,
- a longer context helps encoder-only models up to about 2048 steps,
- decoder-only models are cheaper to train and to run, encoder-only models reach the best accuracy.

So there is no universal patch size: it depends on the architecture, the horizon and the budget.

We cannot change the patch of a pretrained model, but we can change what it covers.
On data resampled every 4 hours, a patch of 32 steps covers 128 hours instead of 32.

<div class="alert alert-success">
    <b>EXERCISE</b>: change what a patch covers
    <ul>
    <li>Resample the counter to 1, 2 and 4 hours with <code>.resample(...).sum()</code>.</li>
    <li>At each resolution, forecast one week with t0 from the same 2048 hours of history.</li>
    <li>Sum the hourly and 2-hourly medians over blocks of 4 hours, and compare the three forecasts on
    the 4-hourly truth with the mean absolute error.</li>
    <li>Which resolution gives the best forecast of the 4-hourly totals?</li>
    </ul>
</div>

Solution: `solutions/forecasting-2_resolution.py`

In [ ]:
errors = {}
for hours in [1, 2, 4]:
    ...

## 7. Multivariate: both directions of a street

Real forecasting problems rarely come as one isolated series.
Boulevard de Sébastopol has one counter per direction. The two series share the weather and the holidays,
but their commute peaks are at opposite times of the day.
A *multivariate* model forecasts them jointly, so each one can use the past of the other.

t0, Chronos-2 and Timestral attend over the variates as well as over time.
They take an array `(n_series, n_variates, n_timesteps)`.
To forecast the variates *independently*, we stack them as separate series instead.

In [ ]:
street = ["Totem 73 boulevard de Sébastopol N-S", "Totem 73 boulevard de Sébastopol S-N"]
counts.loc["2025-06-02":"2025-06-08", street].plot(figsize=(12, 3));

In [ ]:
contexts, futures = tu.make_windows(counts[street], origins, context_length, horizon)
n_origins, n_variates, _ = contexts.shape

results = {}
for model in [t0, chronos2, timestral]:
    independent = model.predict(contexts.reshape(n_origins * n_variates, -1), horizon)
    joint = model.predict(contexts, horizon)
    results[model.name, "independent"] = tu.score(futures, independent.reshape(joint.shape), contexts)
    results[model.name, "joint"] = tu.score(futures, joint, contexts)
pd.DataFrame(results).T

<div class="alert alert-success">
    <b>QUESTION</b>:
    <ul>
    <li>Does the joint forecast help here? The two series already have a long history each.</li>
    <li>In which situations would you expect the other variates to help more?</li>
    </ul>
</div>

## 8. Covariates known in advance: the calendar

Some variables are known over the horizon: the hour, the day of the week, the public and school holidays.
They are *known-future covariates*. A univariate model can only guess a holiday from the past values,
which do not announce it.

`tu.calendar_features` builds these four covariates, in Paris time. The models take them as `future_covariates`,
shaped `(n_series, n_covariates, context + horizon)`.
TabICL turns forecasting into a regression on such features, so the covariates are just extra columns.

In [ ]:
tu.calendar_features(counts.index).loc["2025-03-31":"2025-06-15", ["holiday", "school holiday"]].plot(subplots=True, figsize=(12, 3));

A TSFM learns the effect of a covariate *in context*: it must see a few holidays in the context to
know what a holiday does. With 2048 hours, the context covers less than three months.
At a resolution of 4 hours, 2048 steps cover 11 months, so the context holds the holidays of the past year.

We compare the two resolutions, with no covariates, with the public holidays, and with the school holidays too.
We forecast every week of 2025, and group the weeks by their working days:
a public holiday, a school holiday, or neither. A week can be in the first two groups.
The MASE is computed at each resolution, so compare the rows within a column.

In [ ]:
week_origins = pd.date_range("2025-01-06", "2025-12-22", freq="7D")[::every]
days = tu.calendar_features(pd.date_range(week_origins[0], week_origins[-1] + pd.Timedelta(days=6), freq="D"))
working_days = days[days["weekday"] < 5]
weeks = working_days.groupby(working_days.index.to_period("W"))[["holiday", "school holiday"]].max().astype(bool)
weeks = weeks.loc[week_origins.to_period("W")]
groups = {"public holiday": weeks["holiday"].to_numpy(), "school holiday": weeks["school holiday"].to_numpy(),
          "neither": ~weeks.any(axis=1).to_numpy()}
{group: mask.sum() for group, mask in groups.items()}

In [ ]:
covariates = {
    "none": [],
    "public holidays": ["hour", "weekday", "holiday"],
    "+ school holidays": ["hour", "weekday", "holiday", "school holiday"],
}

results = {}
for hours, n_context in [(1, 2048), (4, 2048)]:
    target = counts[[counter]].resample(f"{hours}h").sum(min_count=hours)
    calendar = tu.calendar_features(target.index)
    n_steps, setting = tu.WEEK // hours, f"{hours}h, {n_context} steps"
    contexts, futures = tu.make_windows(target, week_origins, n_context, n_steps)
    contexts, futures = contexts[:, 0], futures[:, 0]
    for name, columns in covariates.items():
        kwargs = {}
        if columns:
            past, ahead = tu.make_windows(calendar[columns], week_origins, n_context, n_steps)
            kwargs["future_covariates"] = np.concatenate([past, ahead], axis=-1)
        for model in [t0, chronos2, timestral, tabicl]:
            dates = {"origins": week_origins, "freq": f"{hours}h"} if model is tabicl else {}
            scores = tu.mase(futures, model.predict(contexts, n_steps, **kwargs, **dates), contexts, season=n_steps)
            for group, mask in groups.items():
                results[model.name, name, setting, group] = np.nanmean(scores[mask])

results = pd.Series(results).unstack([2, 3]).reindex(list(covariates), level=1)
results.rename_axis(["model", "covariates"]).style.format("{:.2f}")

Easter Monday, with t0 at a resolution of 4 hours and a context of 11 months:

In [ ]:
easter = pd.Timestamp("2025-04-21")
past, ahead = tu.make_windows(calendar[covariates["public holidays"]], [easter], n_context, n_steps)
contexts, futures = tu.make_windows(target, [easter], n_context, n_steps)
index = pd.date_range(end=easter + pd.Timedelta(hours=tu.WEEK - 4), periods=n_context + n_steps, freq="4h")
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, sharey=True)
tu.plot_forecast(index, contexts[0, 0], futures[0, 0], t0.predict(contexts[:, 0], n_steps)[0],
                 ax=axes[0], label="t0", n_context=2 * n_steps)
q = t0.predict(contexts[:, 0], n_steps, future_covariates=np.concatenate([past, ahead], axis=-1))
tu.plot_forecast(index, contexts[0, 0], futures[0, 0], q[0], ax=axes[1], label="t0 + public holidays", n_context=2 * n_steps)
axes[0].set_title("Easter week 2025: Monday 21 April is a public holiday");

<div class="alert alert-success">
    <b>QUESTION</b>:
    <ul>
    <li>Which models use the holidays with 2048 hours of context? Which ones need the longer context?</li>
    <li>In 2025, the spring school holidays of Paris started a week before Easter Monday.
    Can a model separate the effect of the two flags with one example of each?</li>
    <li>A holiday is shorter than a patch of t0 at the hourly resolution. Could this matter?</li>
    </ul>
</div>

## 9. Covariates known only in the past: the weather

Fewer people ride in the rain. The weather is a natural covariate, but it is not known in advance.
There are two honest ways to use it, and one dishonest one:

- as a **past covariate**: the model sees the past weather only, and learns how it relates to the counts;
- as a **future covariate built from weather forecasts**, which have their own errors;
- as a **future covariate built from the observed weather**: it uses information that is not
  available at forecast time. This is *leakage*. It gives an upper bound on what a perfect weather
  forecast would bring, never an estimate of the real performance.

Our file holds the observed weather, so we compare the first and the third option, on top of the calendar.
t0 forecasts past covariates jointly with the target.
Chronos-2 and Timestral take them as covariates with no future values.

In [ ]:
weather = tu.load_weather().reindex(counts.index)[["temperature", "precipitation"]]
weather.loc["2025-06-02":"2025-06-15"].plot(subplots=True, figsize=(12, 4));

In [ ]:
contexts, futures = tu.make_windows(counts[[counter]], origins, context_length, horizon)
contexts, futures = contexts[:, 0], futures[:, 0]
calendar_windows = np.concatenate(tu.make_windows(tu.calendar_features(counts.index), origins, context_length, horizon), axis=-1)
weather_past, weather_future = tu.make_windows(weather, origins, context_length, horizon)
weather_windows = np.concatenate([weather_past, weather_future], axis=-1)

results = {}
for model in [t0, chronos2, timestral]:
    q = model.predict(contexts, horizon, future_covariates=calendar_windows)
    results[model.name, "calendar"] = tu.score(futures, q, contexts)
    q = model.predict(contexts, horizon, past_covariates=weather_past, future_covariates=calendar_windows)
    results[model.name, "calendar + past weather"] = tu.score(futures, q, contexts)
    q = model.predict(contexts, horizon, future_covariates=np.concatenate([calendar_windows, weather_windows], axis=1))
    results[model.name, "calendar + observed future weather (leakage)"] = tu.score(futures, q, contexts)
pd.DataFrame(results).T

A covariate can improve a score for the wrong reason: the model may use it as noise,
or it may leak information. A *placebo* test replaces the covariate with one that has the same
distribution but no relation to the target. If the placebo also improves the score, be suspicious.

<div class="alert alert-success">
    <b>EXERCISE</b>: placebo test
    <ul>
    <li>Build a placebo weather: shuffle the days of <code>weather</code>, keeping the 24 hours of each
    day together, with <code>np.random.default_rng(0)</code>.</li>
    <li>Score t0 with the calendar and the placebo weather as future covariates.</li>
    <li>Compare with the table above. What does the placebo tell you about the weather covariate?</li>
    </ul>
</div>

Solution: `solutions/forecasting-3_placebo-weather.py`

In [ ]:
rng = np.random.default_rng(0)
placebo = ...

## 10. Many sites: every counter of Paris

The city needs a forecast for each of its counters. A TSFM forecasts all of them in one batch,
with the same model and no training. We load every counter for 2024 and 2025,
and keep the ones with almost no missing hours in 2025.

In [ ]:
panel = tu.load_bike_counts(tu.SITES_DATA)
missing = panel.loc["2025"].isna().mean()
sites = panel.loc[:, missing < 0.02]
sites.shape

Chronos-2 can also share information *across* series: with `cross_learning=True`, all the series of a
batch attend to each other. This helps most when each series has a short history,
for instance a counter that was just installed.

Beware of the batches. If two origins sit in the same batch, the context of the later one contains the
future of the earlier one: cross-learning would leak it. We forecast each origin in its own batch.

In [ ]:
site_origins = pd.date_range("2025-01-06", "2025-12-15", freq="42D")[::every]
results, mase = {}, {}
for n_context in [168, 2048]:
    for name, model, kwargs in [("Seasonal naive", models["Seasonal naive"], {}),
                                ("t0", t0, {}),
                                ("Chronos-2", chronos2, {}),
                                ("Chronos-2, cross-learning", chronos2, {"cross_learning": True})]:
        scores = []
        for origin in site_origins:
            contexts, futures = tu.make_windows(sites, [origin], n_context, horizon)
            # Scale the errors with the full context, whatever the context given to the model
            full_contexts, _ = tu.make_windows(sites, [origin], 2048, horizon)
            q = model.predict(contexts[0], horizon, **kwargs)
            scores.append(tu.mase(futures[0], q, full_contexts[0]))
        mase[name, n_context] = np.nanmean(scores, axis=0)
        results[name, n_context] = np.nanmean(scores)
pd.Series(results).unstack().rename_axis(index="model", columns="context length")

The map shows the MASE of t0 at each counter. Click on a marker for its name.

In [ ]:
import folium

locations = tu.load_counter_locations().loc[sites.columns]
site_mase = pd.Series(mase["t0", 2048], index=sites.columns)
m = folium.Map(location=locations.mean().tolist(), zoom_start=12)
for name, (lat, lon) in locations.iterrows():
    color = "green" if site_mase[name] < 0.8 else "orange" if site_mase[name] < 1.2 else "red"
    folium.CircleMarker([lat, lon], radius=6, color=color, fill=True,
                        popup=f"{name}: MASE {site_mase[name]:.2f}").add_to(m)
m

<div class="alert alert-success">
    <b>EXERCISE</b>: where does the model fail?
    <ul>
    <li>Find the three counters with the largest MASE for t0.</li>
    <li>Plot the last two weeks of their context and their forecast at the first origin.</li>
    <li>What happens at these counters? Could any model have forecast it?</li>
    </ul>
</div>

Solution: `solutions/forecasting-4_worst-counters.py`

In [ ]:
worst = ...

## 11. A year and more of history

A holiday comes back once a year. To see the last one, an hourly model needs more than 8760 steps of context.
The models accept very different contexts: TiRex-2 keeps the last 2048 steps, Chronos-2 the last 8192,
TimesFM 3 up to 15360 and Timestral up to 32768. A longer context is cut without a warning.

We forecast every week of 2025 again, from 2048, 8192 and 16384 hours of history (16384 hours is almost two years).
The MASE is scaled with the same 2048 hours for every context, so that the columns compare.
A long context takes memory: `tu.predict_in_chunks` forecasts 8 weeks at a time.

In [ ]:
lengths = [2048, 8192, 16384] if GPU else [2048, 8192]
target = counts[[counter]]
calendar = tu.calendar_features(counts.index)
scale_contexts, futures = tu.make_windows(target, week_origins, 2048, horizon)
scale_contexts, futures = scale_contexts[:, 0], futures[:, 0]
runs = [("Chronos-2", chronos2, False), ("TiRex-2", models["TiRex-2"], False), ("t0", t0, False),
        ("TimesFM 3", models["TimesFM 3"], False), ("TimesFM 3 + calendar", models["TimesFM 3"], True),
        ("Timestral", timestral, False)]
if GPU:  # a few minutes on a GPU, too long on a CPU
    runs.append(("TabICL + calendar", tu.TabICL(max_context=16384), True))

results = {}
for n_context in lengths:
    contexts, _ = tu.make_windows(target, week_origins, n_context, horizon)
    past, ahead = tu.make_windows(calendar, week_origins, n_context, horizon)
    for name, model, use_calendar in runs:
        kwargs = {"future_covariates": np.concatenate([past, ahead], axis=-1)} if use_calendar else {}
        if model.name == "TabICL":
            kwargs["origins"] = week_origins
        q = tu.predict_in_chunks(model, contexts[:, 0], horizon, **kwargs)
        scores = tu.mase(futures, q, scale_contexts)
        for group, mask in {"all weeks": np.ones(len(scores), bool), **groups}.items():
            results[name, group, n_context] = np.nanmean(scores[mask])

results = pd.Series(results).unstack()
results.style.format("{:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
for ax, group in zip(axes, ["all weeks", *groups]):
    results.xs(group, level=1).T.plot(ax=ax, marker="o", logx=True, title=group, legend=ax is axes[0])
    ax.axvline(8760, color="k", ls=":")
    ax.set(xlabel="context (hours), dotted: one year", ylabel="MASE")
plt.tight_layout()

<div class="alert alert-success">
    <b>QUESTION</b>:
    <ul>
    <li>Which models gain from more than a year of history? On which weeks?</li>
    <li>Why are some curves flat beyond 2048 or 8192 hours?</li>
    <li>Section 8 showed a year of history at a resolution of 4 hours, with 2048 steps.
    Which is cheaper, and which forecasts better?</li>
    </ul>
</div>

## Take-aways and open problems

- A TSFM forecasts a new series with no training, from a context of past values.
  Behind the same interface, the models differ by their tokens, their backbone and the way they produce the horizon.
- Evaluate on many origins, with a probabilistic metric (WQL) as well as a point one (MASE),
  check the calibration at every lead time, and always compare with a seasonal naive baseline.
- Rolling out a direct model degrades the median and, more, the uncertainty:
  feeding back the median makes the intervals too narrow.
- The patch size trades resolution against cost, and its best value depends on the architecture and the horizon.
- A model learns the effect of a covariate in context: the context must hold examples of it.
  Measure the gain on the weeks where it matters, and test it with a placebo.
- Covariates must be available at forecast time. Observed future weather is leakage.
- One model forecasts every site with no training. A long context matters more than cross-learning,
  which is meant for short histories, and its batches must respect time.
- More than a year of history lets some models see the last holidays. Others cut the context without a warning:
  check the maximum context of a model before feeding it years of data.

Open questions for the field:

- Which tokenization? There is no universal patch size.
- When do the variates and covariates help? Gains are inconsistent across benchmarks.
- How to evaluate fairly, when the pretraining corpora may contain the benchmark series,
  and every new model claims the top of a leaderboard?
- How to forecast hierarchies (a street, a district, the city) with coherent sums?
- How to react to a regime change, such as a lockdown, that the context does not announce?